# PixCell Stage 3 Ablation Notebook for Colab A100

This notebook runs the paired Stage 3 ablation cache generation to `5000` tiles, zips the result, uploads it to S3, and can optionally continue to the unpaired run in the same Colab session.

Recommended settings on a single A100:
- `DEVICE = "cuda"`
- `JOBS = 1`
- `NUM_STEPS = 20`

The main bottleneck is GPU inference.

In [ ]:
import os
import subprocess
import zipfile
from pathlib import Path

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

print(f'IN_COLAB={IN_COLAB}')

## Clone Repo and Install Dependencies

In [ ]:
%cd /content
!rm -rf /content/PixCell
!git clone https://github.com/pohaoc2/PixCell.git
%cd /content/PixCell
!git checkout main
!python -m pip install --upgrade pip setuptools==75.2.0 wheel
!python -m pip install mmcv==1.7.0 --no-build-isolation
!python -m pip install -r requirements.txt
!python -m pip install awscli
!nvidia-smi

## Helpers

In [ ]:
REPO = Path('/content/PixCell')

def run(cmd, cwd=REPO):
    print(' '.join(cmd))
    return subprocess.run(cmd, cwd=str(cwd), check=True)

def load_secret(name):
    if not IN_COLAB or userdata is None:
        return None
    try:
        value = userdata.get(name)
    except Exception:
        value = None
    if value:
        os.environ[name] = value
    return value

def pull_s3_path(source, dest):
    source = str(source)
    dest = Path(dest)
    if source.lower().endswith('.zip'):
        dest.parent.mkdir(parents=True, exist_ok=True)
        local_zip = dest.parent / Path(source).name
        run(['aws', 's3', 'cp', source, str(local_zip)])
        run(['unzip', '-q', '-o', str(local_zip), '-d', str(dest.parent)])
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        run(['aws', 's3', 'sync', source, str(dest)])

def zip_tree(source_dir, zip_path):
    source_dir = Path(source_dir)
    zip_path = Path(zip_path)
    zip_path.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_STORED) as zf:
        for path in sorted(source_dir.rglob('*')):
            if path.is_file():
                zf.write(path, arcname=str(Path(source_dir.name) / path.relative_to(source_dir)))
    print(f'Wrote zip: {zip_path}')
    print(f'Size (GB): {zip_path.stat().st_size / (1024 ** 3):.2f}')

## Load Secrets

Expected Colab secrets:
- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`
- `HF_TOKEN`

In [ ]:
for key in ['AWS_ACCESS_KEY_ID', 'AWS_SECRET_ACCESS_KEY', 'AWS_DEFAULT_REGION', 'HF_TOKEN']:
    value = load_secret(key)
    print(f'{key}:', 'set' if value else 'missing')

## Config

In [ ]:
REPO = Path('/content/PixCell')
DATA_ROOT = REPO / 'data' / 'orion-crc33'
CHECKPOINT_DIR = REPO / 'checkpoints' / 'pixcell_controlnet_exp' / 'npy_inputs'
PRETRAINED_MODELS_DIR = REPO / 'pretrained_models'
OUTPUT_DIR = REPO / 'inference_output' / 'paired_ablation' / 'ablation_results'

S3_DATA_ROOT = 's3://bagherilab-working/pohao/share_space/orion-crc33.zip'
S3_CHECKPOINT_ROOT = 's3://bagherilab-working/pohao/share_space/checkpoints/npy_inputs/'
S3_PRETRAINED_MODELS_ROOT = None
S3_OUTPUT_ROOT = None
S3_ZIP_DEST = 's3://bagherilab-working/pohao/share_space/ablation_generation/paired/'

ZIP_OUTPUT = REPO / 'inference_output' / 'paired_ablation_5000_tiles.zip'

TARGET_TOTAL_TILES = 5000
CHUNK_TARGETS = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]

GUIDANCE_SCALE = 2.5
NUM_STEPS = 20
SEED = 42
TILE_SAMPLE_SEED = 42
DEVICE = 'cuda'
JOBS = 1

print(DATA_ROOT)
print(CHECKPOINT_DIR)
print(PRETRAINED_MODELS_DIR)
print(OUTPUT_DIR)

## Pull Data and Checkpoints from S3

In [ ]:
if S3_DATA_ROOT:
    pull_s3_path(S3_DATA_ROOT, DATA_ROOT)

if S3_CHECKPOINT_ROOT:
    pull_s3_path(S3_CHECKPOINT_ROOT, CHECKPOINT_DIR)

if S3_PRETRAINED_MODELS_ROOT:
    pull_s3_path(S3_PRETRAINED_MODELS_ROOT, PRETRAINED_MODELS_DIR)


## Download Pretrained Models from Hugging Face

Run this if `pretrained_models/` is not already available from S3.

In [ ]:
hf_token = os.environ.get('HF_TOKEN')
if hf_token:
    run([
        'python', 'stage0_setup.py',
        '--save-dir', str(PRETRAINED_MODELS_DIR),
        '--model', 'all',
        '--token', hf_token,
    ])
else:
    print('HF_TOKEN is not set; skipping model download.')

## Full Paired Run to 5000 Tiles

In [ ]:
for target in CHUNK_TARGETS:
    if target > TARGET_TOTAL_TILES:
        continue
    print(f'\n=== Growing paired cache to {target} tiles ===')
    run([
        'python', '-m', 'tools.stage3.generate_ablation_subset_cache',
        '--data-root', str(DATA_ROOT),
        '--checkpoint-dir', str(CHECKPOINT_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--target-total-tiles', str(target),
        '--seed', str(SEED),
        '--tile-sample-seed', str(TILE_SAMPLE_SEED),
        '--device', DEVICE,
        '--guidance-scale', str(GUIDANCE_SCALE),
        '--num-steps', str(NUM_STEPS),
        '--jobs', str(JOBS),
    ])
    if S3_OUTPUT_ROOT:
        run(['aws', 's3', 'sync', str(OUTPUT_DIR), S3_OUTPUT_ROOT])

## Check Paired Progress

In [ ]:
tile_dirs = sorted([p for p in OUTPUT_DIR.iterdir() if p.is_dir() and (p / 'manifest.json').exists()]) if OUTPUT_DIR.exists() else []
print('paired cached tiles:', len(tile_dirs))
if tile_dirs:
    print('first tile:', tile_dirs[0].name)
    print('last tile:', tile_dirs[-1].name)

## Zip and Upload Paired Output

In [ ]:
zip_tree(OUTPUT_DIR, ZIP_OUTPUT)

In [ ]:
run(['aws', 's3', 'cp', str(ZIP_OUTPUT), S3_ZIP_DEST])

## Optional: Continue to Unpaired in the Same Notebook

In [ ]:
UNPAIRED_MAPPING_JSON = REPO / 'inference_output' / 'unpaired_ablation' / 'metadata' / 'unpaired_mapping.json'
UNPAIRED_OUTPUT_DIR = REPO / 'inference_output' / 'unpaired_ablation' / 'ablation_results'
UNPAIRED_ZIP_OUTPUT = REPO / 'inference_output' / 'unpaired_ablation_5000_tiles.zip'
S3_UNPAIRED_ZIP_DEST = 's3://bagherilab-working/pohao/share_space/ablation_generation/unpaired/'
UNPAIRED_TARGET_TOTAL_TILES = 5000
UNPAIRED_CHUNK_TARGETS = [500, 1000, 1500, 2000, 2500, 3000, 3500, 4000, 4500, 5000]
UNPAIRED_JOBS = 1


In [ ]:
run([
    'python', '-m', 'tools.stage3.prepare_unpaired_ablation_dataset',
    '--paired-cache-root', str(OUTPUT_DIR),
    '--data-root', str(DATA_ROOT),
    '--metadata-only',
    '--mapping-output', str(UNPAIRED_MAPPING_JSON),
    '--seed', str(SEED),
])

In [ ]:
for target in UNPAIRED_CHUNK_TARGETS:
    if target > UNPAIRED_TARGET_TOTAL_TILES:
        continue
    print(f'\n=== Growing unpaired cache to {target} tiles ===')
    run([
        'python', '-m', 'tools.stage3.generate_ablation_subset_cache',
        '--data-root', str(DATA_ROOT),
        '--style-mapping-json', str(UNPAIRED_MAPPING_JSON),
        '--checkpoint-dir', str(CHECKPOINT_DIR),
        '--output-dir', str(UNPAIRED_OUTPUT_DIR),
        '--target-total-tiles', str(target),
        '--seed', str(SEED),
        '--tile-sample-seed', str(TILE_SAMPLE_SEED),
        '--device', DEVICE,
        '--guidance-scale', str(GUIDANCE_SCALE),
        '--num-steps', str(NUM_STEPS),
        '--jobs', str(UNPAIRED_JOBS),
    ])

In [ ]:
zip_tree(UNPAIRED_OUTPUT_DIR, UNPAIRED_ZIP_OUTPUT)
run(['aws', 's3', 'cp', str(UNPAIRED_ZIP_OUTPUT), S3_UNPAIRED_ZIP_DEST])